# 01 Importing Libraries

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import os

# 02 Import Data

In [5]:
path = r'C:\Users\isava\OneDrive\Documents\CareerFoundry\Data Immersion\PythonFundamentals\Instacart Basket Analysis'
path

'C:\\Users\\isava\\OneDrive\\Documents\\CareerFoundry\\Data Immersion\\PythonFundamentals\\Instacart Basket Analysis'

In [7]:
df_ords = pd.read_csv(os.path.join(path,'02 Data', 'Prepared Data','orders_wrangled.csv'), index_col = False)
df_prods = pd.read_csv(os.path.join(path,'02 Data', 'Original Data','products.csv'), index_col = False)

# 03 Data Consistency Checks

In [44]:
df_ords.describe()

,Unnamed: 0,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_users_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710541e+06,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552705e+05,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710541e+06,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421082e+06,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


Order day of week and order hour of day are index based and range from 0-6 and 0-23 accordingly. The maximum days since prior order is 30 days. The order and user id have a wide range but order number caps at 100. Perhaps we can expect repeat order_numbers per customer or this need to be raised to avoid duplicates that would become uninformative. 

In [9]:
df_prods.describe()

,product_id,aisle_id,department_id,prices
count,49693.000000,49693.000000,49693.000000,49693.000000
mean,24844.345139,67.770249,11.728433,9.994136
std,14343.717401,38.316774,5.850282,453.519686
min,1.000000,1.000000,1.000000,1.000000
25%,12423.000000,35.000000,7.000000,4.100000
50%,24845.000000,69.000000,13.000000,7.100000
75%,37265.000000,100.000000,17.000000,11.200000
max,49688.000000,134.000000,21.000000,99999.000000


## Mixed Data Types

In [64]:
for col in df_ords.columns.tolist():
  weird = (df_ords[[col]].map(type) != df_ords[[col]].iloc[0].apply(type)).any(axis = 1)
  if len (df_ords[weird]) > 0:
    print (col)

There appears to be no mixed data types in df_ords.

In [60]:
df_ords.dtypes

Unnamed: 0                        int64
order_id                          int64
user_id                           int64
order_number                      int64
orders_day_of_week                int64
order_hour_of_day                 int64
days_since_users_prior_order    float64
dtype: object

## Missing Values

### Products

In [25]:
df_prods.isnull().sum()

product_id        0
product_name     16
aisle_id          0
department_id     0
prices            0
dtype: int64

In [16]:
df_nan = df_prods[df_prods['product_name'].isnull() == True]
df_nan

,product_id,product_name,aisle_id,department_id,prices
33,34,NaN,121,14,12.2
68,69,NaN,26,7,11.8
115,116,NaN,93,3,10.8
261,262,NaN,110,13,12.1
525,525,NaN,109,11,1.2
1511,1511,NaN,84,16,14.3
1780,1780,NaN,126,11,12.3
2240,2240,NaN,52,1,14.2
2586,2586,NaN,104,13,12.4
3159,3159,NaN,126,11,13.1


In [18]:
df_prods.shape

(49693, 5)

In [13]:
df_prods_clean = df_prods[df_prods['product_name'].isnull() == False]
df_prods_clean.shape

(49677, 5)

### Orders

In [73]:
df_ords.isnull().sum()

Unnamed: 0                           0
order_id                             0
user_id                              0
order_number                         0
orders_day_of_week                   0
order_hour_of_day                    0
days_since_users_prior_order    206209
dtype: int64

There are a lot of missing values in the days_since_users_prior_order column. There may be multiple orders where its the first order and there hasnt been a subsequant order. Therefore a 'missing' prior order is likely valid. 

In [77]:
df_ords_nan = df_ords[df_ords['days_since_users_prior_order'].isnull() == True]
df_ords_nan

,Unnamed: 0,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_users_prior_order
0,0,2539329,1,1,2,8,NaN
11,11,2168274,2,1,2,11,NaN
26,26,1374495,3,1,1,14,NaN
39,39,3343014,4,1,6,11,NaN
45,45,2717275,5,1,3,12,NaN
...,...,...,...,...,...,...,...
3420930,3420930,969311,206205,1,4,12,NaN
3420934,3420934,3189322,206206,1,3,18,NaN
3421002,3421002,2166133,206207,1,6,19,NaN
3421019,3421019,2227043,206208,1,1,15,NaN


It appears that a lot of these have an order number of 1. So I will check is all of them are 1.

In [81]:
df_ords_nan.shape

(206209, 7)

In [87]:
df_ords_firstorder = df_ords[df_ords['order_number'] == 1]
df_ords_firstorder

,Unnamed: 0,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_users_prior_order
0,0,2539329,1,1,2,8,NaN
11,11,2168274,2,1,2,11,NaN
26,26,1374495,3,1,1,14,NaN
39,39,3343014,4,1,6,11,NaN
45,45,2717275,5,1,3,12,NaN
...,...,...,...,...,...,...,...
3420930,3420930,969311,206205,1,4,12,NaN
3420934,3420934,3189322,206206,1,3,18,NaN
3421002,3421002,2166133,206207,1,6,19,NaN
3421019,3421019,2227043,206208,1,1,15,NaN


In [93]:
df_ords_firstorder.shape

(206209, 7)

This has the same dimension as the missing values data frame. Now I will checl if all the values in the missing values data frame have a order number of 1. 

In [96]:
df_ords_nan_firstorder = df_ords_nan[df_ords_nan['order_number'] == 1]
df_ords_nan_firstorder

,Unnamed: 0,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_users_prior_order
0,0,2539329,1,1,2,8,NaN
11,11,2168274,2,1,2,11,NaN
26,26,1374495,3,1,1,14,NaN
39,39,3343014,4,1,6,11,NaN
45,45,2717275,5,1,3,12,NaN
...,...,...,...,...,...,...,...
3420930,3420930,969311,206205,1,4,12,NaN
3420934,3420934,3189322,206206,1,3,18,NaN
3421002,3421002,2166133,206207,1,6,19,NaN
3421019,3421019,2227043,206208,1,1,15,NaN


In [98]:
df_ords_nan_firstorder.shape

(206209, 7)

So all of the Order number 1 have no prior order. Due to this the NaN value is valid and I will keep it for now. 

## Duplicate

### Products

In [15]:
df_dups = df_prods_clean[df_prods_clean.duplicated()]
df_dups

,product_id,product_name,aisle_id,department_id,prices
462,462,Fiber 4g Gummy Dietary Supplement,70,11,4.8
18459,18458,Ranger IPA,27,5,9.2
26810,26808,Black House Coffee Roasty Stout Beer,27,5,13.4
35309,35306,Gluten Free Organic Peanut Butter & Chocolate ...,121,14,6.8
35495,35491,Adore Forever Body Wash,127,11,9.9


In [17]:
df_prods_clean.shape

(49677, 5)

In [19]:
df_prods_clean_no_dups = df_prods_clean.drop_duplicates()
df_prods_clean_no_dups.shape

(49672, 5)

In [21]:
df_prods_clean_no_dups.describe()

,product_id,aisle_id,department_id,prices
count,49672.000000,49672.000000,49672.000000,49672.000000
mean,24850.349775,67.762442,11.728942,9.993282
std,14340.705287,38.315784,5.850779,453.615536
min,1.000000,1.000000,1.000000,1.000000
25%,12432.750000,35.000000,7.000000,4.100000
50%,24850.500000,69.000000,13.000000,7.100000
75%,37268.250000,100.000000,17.000000,11.100000
max,49688.000000,134.000000,21.000000,99999.000000


In [16]:
df_snacks =  df_prods[df_prods['department_id']==19]
df_snacks.head()

,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.8
15,16,Mint Chocolate Flavored Syrup,103,19,5.2
24,25,Salted Caramel Lean Protein & Fiber Bar,3,19,1.9
31,32,Nacho Cheese White Bean Chips,107,19,4.9
40,41,Organic Sourdough Einkorn Crackers Rosemary,78,19,6.5


In [17]:
df_snacks_3 = df_prods.loc[df_prods['department_id'].isin([19])]

### Orders

In [107]:
df_ords_dups = df_ords[df_ords.duplicated()]
df_ords_dups

,Unnamed: 0,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_users_prior_order


There are no duplicates found. 

# 04 Export Wrangled Data Frame

In [114]:
df_prods_clean_no_dups.to_csv(os.path.join(path, '02 Data','Prepared Data', 'products_checked.csv'))
#No changes to df_ords made.